# GLips Lippenlesen mit Deep Learning

## Datenimport
Der Datensatz besteht aus Videos aus dem Hessischen Parlament wobei 500 Verschiedene Wörter enthalten sind. Die Videos sind bereits auf die Lippen gecropt und in 25 Frames pro Sekunde umgewandelt worden. Der Folder ist in Wörter unterteilt, die jeweils train test und validation Ordner enthalten.

In [1]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.io as io

class GLipsFullClipDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []

        self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        for cls_name in self.classes:
            split_folder = os.path.join(root_dir, cls_name, 'test')
            if os.path.exists(split_folder):
                for file in os.listdir(split_folder):
                    if file.endswith('.mp4'):
                        self.samples.append((os.path.join(split_folder, file), self.class_to_idx[cls_name]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]

        # Load entire video: (T, C, H, W)
        video, _, _ = io.read_video(video_path, pts_unit='sec', output_format='TCHW')

        video = video.float() / 255.0

        if self.transform:
            video = self.transform(video)

        # Returns (C, T, H, W) and label
        return video.permute(1, 0, 2, 3), label

In [2]:
dataset = GLipsFullClipDataset(root_dir='./GLips/lipread_files/')
dataloader = DataLoader(dataset, batch_size=5, shuffle=True)

FileNotFoundError: [Errno 2] No such file or directory: './GLips/lipread_files/'

## Modell
in README.md können sie sich die Modellarchitektur anschauen. Es handelt sich um ein 3D-CNN mit mehreren Convolutional und Pooling Schichten, gefolgt von einem Transformer und einem Neuralen Netzwerk. Das Modell ist darauf ausgelegt, die zeitlichen und räumlichen Merkmale der Lippenbewegungen zu erfassen.

### 3D CNN + ResNet
Mit dem 3DCNN werden die räumlichen und zeitlichen Merkmale der Lippenbewegungen extrahiert. Die Convolutional Schichten erfassen lokale und kleine zeitliche Muster. Der ResNet-Block ermöglicht es, detailierte Merkmale der Lippenbewegungen zu erfassen, indem er die Informationen über mehrere Schichten hinweg weitergibt. Dies hilft, das Problem des Vanishing Gradient zu vermeiden und ermöglicht es dem Modell, tiefere Netzwerke zu trainieren.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class CNN3D(nn.Module):
    def __init__(self, in_channels=3, out_channels=512):
        super(CNN3D, self).__init__()

        # 3D Conv Layer
        self.frontend3D = nn.Sequential(
            nn.Conv3d(3, 64, kernel_size=(5, 7, 7), stride=(1, 2, 2), padding=(2, 3, 3), bias=False),
            nn.BatchNorm3d(64),
            nn.ReLU(True),
            nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1))
        )

        # 2D Backend (ResNet-18)
        resnet = models.resnet18(weights=None)

        # Da das Frontend bereits 64 Channels ausgibt, ersetzen wir resnet.conv1
        self.resnet2d = nn.Sequential(*list(resnet.children())[4:-2])

        # Global Pooling, um H und W zu eliminieren
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        # x shape: (B, 3, T, 96, 96)
        B, C, T, H, W = x.size()

        # Frontend 3D
        x = self.frontend3D(x)

        # Neuer Shape nach Strides: (B, 64, T, H', W')
        # Um 2D ResNet auf jeden Frame anzuwenden, mischen wir B und T
        x = x.transpose(1, 2).contiguous() # (B, T, 64, H', W')
        x = x.view(-1, 64, x.size(3), x.size(4)) # (B*T, 64, H', W')

        # ResNet Features extrahieren
        x = self.resnet2d(x) # (B*T, 512, H'', W'')
        x = self.avgpool(x) # (B*T, 512, 1, 1)
        x = x.view(B, T, 512) # Zurück in die Sequenzform (B, T, D)

        return x # Output: (B, T, D)

## Transformer
Der Transformer-Teil des Modells ist dafür verantwortlich, die zeitlichen Abhängigkeiten in den extrahierten Merkmalen der Lippenbewegungen zu erfassen. Er besteht aus mehreren Schichten von Multi-Head Attention und Feedforward-Netzwerken, die es dem Modell ermöglichen, komplexe zeitliche Muster zu lernen und die relevanten Informationen über die gesamte Sequenz hinweg zu integrieren.

In [ ]:
import torch
import torch.nn as nn
import math

class LipreadingTransformer(nn.Module):
    def __init__(self, d_model=512, nhead=8, num_layers=6, dim_feedforward=2048, dropout=0.1):
        super(LipreadingTransformer, self).__init__()

        # Positional Encoding, damit das Modell weiß, welcher Frame zuerst kommt
        self.pos_encoder = PositionalEncoding(d_model, dropout)

        # Transformer Encoder Layer & Block
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True  # Wichtig, damit wir (B, T, D) nutzen können
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)

        self.d_model = d_model

    def forward(self, x):
        # x kommt vom CNN mit Shape: (B, T, D)
        x = x * math.sqrt(self.d_model) # Skalierung für Stabilität
        x = self.pos_encoder(x)

        # Self-Attention über die Zeitachse T
        output = self.transformer_encoder(x) # Output Shape: (B, T, D)

        return output

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=500):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

## Orchestrator und Linear
Hier werden die verschiedenen Komponenten des Modells zusammengeführt. Der Orchestrator verbindet das 3D-CNN mit dem Transformer, um die extrahierten Merkmale der Lippenbewegungen zu verarbeiten. Am Ende des Orchestrators befindet sich eine lineare Schicht, die die Ausgabe des Transformers in die gewünschte Anzahl von Klassen (Wörtern) umwandelt, um die Vorhersage zu ermöglichen.

In [ ]:
class GLipsModel(nn.Module):
    def __init__(self, num_classes=500):
        super(GLipsModel, self).__init__()
        self.cnn = CNN3D()
        self.transformer = LipreadingTransformer(d_model=512)

        # Finales Netzwerk für die Vorhersage
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x):
        # 1. 3D-CNN + ResNet Extraktion
        # Input: (B, 3, T, 96, 96) -> Output: (B, T, 512)
        x = self.cnn(x)

        # 2. Transformer Context
        # Output: (B, T, 512)
        x = self.transformer(x)

        # 3. Mean Pooling über die Zeit (T)
        # Output: (B, 512)
        x = torch.mean(x, dim=1)

        # 4. Classification Head
        # Output: (B, 500)
        logits = self.classifier(x)

        return logits

## Training

In [ ]:
Model = GLipsModel()

optimizer = torch.optim.Adam(Model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Model.to(device)

num_epochs = 100

for epoch in num_epochs:
    optimizer.zero_grad()
    for batch_idx, (data, target) in enumerate(dataloader):
        data, target = data.to(device), target.to(device)
        output = Model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")